# Step 5 — Build the published-method research dataset
Run **Runtime → Run all**, then **File → Save a copy in GitHub** with this filename. Earlier notebooks do not need to run.

We adopt the published daily Kalshi measures from Diercks, Katz and Wright and merge them with SF Fed USMPD statement-window two-year Treasury yield changes. This is an auditable candidate dataset, not completed forecasting research.

**Fixed choices:** scheduled meetings from January 2022; exact previous-weekday UTC date; no older-date substitution; one row per event; exclude missing or unreconciled measurements. Units: standard deviation and Treasury moves in basis points. No outcomes enter inclusion choices other than availability.

**What reliability means here:** we check schemas, unique keys, probability totals, variance and mean reconciliation, pre-announcement date ordering and join cardinality. This cannot certify raw quote freshness or the authors' retrospective processing. Their daily method carries forward prices, corrects monotonicity, moves some probability masses and normalizes probabilities. These assumptions remain visible.

**Sources:** [replication package](https://github.com/jdkatz21/Prediction_Markets_Public), [distribution construction](https://github.com/jdkatz21/Prediction_Markets_Public/blob/main/code/convert_trades_to_pdfs/convert_trade_level_data_cdfs.R), [USMPD definitions](https://www.frbsf.org/research-and-insights/data-and-indicators/us-monetary-policy-event-study-database/).

The earlier 16:00 Eastern cutoff is replaced by the authors' UTC daily convention. Their mean adjustment needs reconciliation with the actual target-rate definition before supporting policy-surprise calculations. Standard deviation is unchanged by a constant location shift.

### 1. Download the published sources and record hashes

In [1]:
import io
import json
import time
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
import numpy as np
import pandas as pd

# Fixed scope. Refreshing the files can update history; hashes identify each download.
STUDY_START = pd.Timestamp("2022-01-01")
AS_OF_DATE = pd.Timestamp.now(tz="America/New_York").tz_localize(None).normalize()
SOURCES = {
  "moments": "https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_moments_data/daily_moments_fed_levels.csv",
  "distributions": "https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_distribution_data/daily_distributions_fed_levels.csv",
  "usmpd": "https://www.frbsf.org/wp-content/uploads/USMPD.xlsx"
}
source_receipts = []
source_bytes = {}

def download(name, url):
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "ResearchProject/1.0"})
            with urllib.request.urlopen(request, timeout=45) as response:
                data = response.read()
                modified = response.headers.get("Last-Modified")
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    source_bytes[name] = data
    source_receipts.append({
        "source": name, "url": url, "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
        "retrieved_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "last_modified_http": modified
    })
    return data

moments = pd.read_csv(io.BytesIO(download("moments", SOURCES["moments"])))
distributions = pd.read_csv(io.BytesIO(download("distributions", SOURCES["distributions"])))
workbook = pd.ExcelFile(io.BytesIO(download("usmpd", SOURCES["usmpd"])))
statements = pd.read_excel(workbook, sheet_name="Statements")
print("Downloaded published moments, distributions and USMPD.")
print("No account credentials or API keys are needed.")


Downloaded published moments, distributions and USMPD.
No account credentials or API keys are needed.


### 2. Validate source structure and construct scheduled meeting keys
USMPD defines the meeting calendar. Kalshi's inferred expiry_date is not used as an official announcement date. Legacy FED and current KXFED identifiers are standardized; duplicate keys stop execution.

In [2]:
required_m = {"date", "contract_preamble", "mean", "variance", "daily_volume"}
required_d = {"date", "contract_preamble", "strike", "probability"}
assert required_m <= set(moments.columns), "Moments schema changed"
assert required_d <= set(distributions.columns), "Distribution schema changed"
assert {"Date", "Unscheduled", "UST2Y"} <= set(statements.columns), "USMPD schema changed"

for frame in [moments, distributions]:
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    assert frame["date"].notna().all()
    frame["event_key"] = frame["contract_preamble"].str.replace(r"^KX", "", regex=True)
assert not moments.duplicated(["event_key", "date"]).any(), "Duplicate event/date moments"
assert not distributions.duplicated(["event_key", "date", "strike"]).any(), "Duplicate bins"
for column in ["mean", "variance", "daily_volume"]:
    moments[column] = pd.to_numeric(moments[column], errors="raise")
for column in ["strike", "probability"]:
    distributions[column] = pd.to_numeric(distributions[column], errors="raise")
assert moments["variance"].dropna().ge(-1e-12).all(), "Negative variance"
assert distributions["probability"].dropna().between(0, 100 + 1e-8).all()

statements["Date"] = pd.to_datetime(statements["Date"], errors="raise")
assert statements["Date"].is_unique
assert statements["Unscheduled"].isin([0, 1]).all()
statements["UST2Y"] = pd.to_numeric(statements["UST2Y"], errors="raise")
# Never use the source's expiry_date as an official meeting date:
# the authors calculate it from the last observed trade day.
events = statements.loc[
    statements["Unscheduled"].eq(0) &
    statements["Date"].ge(STUDY_START) &
    statements["Date"].le(AS_OF_DATE), ["Date", "UST2Y"]
].rename(columns={"Date": "meeting_date", "UST2Y": "ust2y_change_pp"}).copy()
events["event_key"] = events["meeting_date"].map(
    lambda d: "FED-" + d.strftime("%y%b").upper()
)
assert events["event_key"].is_unique, "Two meetings share an event month; inspect mapping"
events["observation_date"] = events["meeting_date"] - pd.offsets.BDay(1)
assert events["observation_date"].lt(events["meeting_date"]).all()
print("Kalshi observation range:", moments["date"].min().date(), "to", moments["date"].max().date())
print("Latest USMPD statement:", statements["Date"].max().date())
print("Scheduled meetings in scope:", len(events))
print("Event mapping uses official USMPD meeting dates and month identifiers.")
print("Previous weekday convention: selected FOMC cutoffs must also be checked for holidays.")


Kalshi observation range: 2021-07-02 to 2026-10-16
Latest USMPD statement: 2026-09-16
Scheduled meetings in scope: 38
Event mapping uses official USMPD meeting dates and month identifiers.
Previous weekday convention: selected FOMC cutoffs must also be checked for holidays.


### 3. Reconcile probability distributions with published moments and audit exclusions
A percentage-point change of 0.05 equals 5 basis points. Published probabilities are percentages and must divide by 100. The mean reconstruction follows the authors' +0.125 adjustment, without treating it as an observed upper-bound rate.

In [3]:
# Join exactly one published daily observation to each scheduled event.
audit = events.merge(
    moments[["event_key", "date", "mean", "variance", "daily_volume"]],
    left_on=["event_key", "observation_date"],
    right_on=["event_key", "date"], how="left", validate="one_to_one",
    indicator="moments_match"
)
assert len(audit) == len(events)
targets = events[["event_key", "observation_date"]]
bins = distributions.merge(
    targets, left_on=["event_key", "date"],
    right_on=["event_key", "observation_date"],
    how="inner", validate="many_to_one"
)
checks = []
for (event_key, date), group in bins.groupby(["event_key", "date"]):
    p = group["probability"].to_numpy(dtype=float) / 100
    x = group["strike"].to_numpy(dtype=float)
    finite = np.isfinite(p).all() and np.isfinite(x).all()
    total = p.sum()
    if finite and total > 0:
        weights = p / total
        mean_unadjusted = np.sum(weights * x)
        var_rebuilt = np.sum(weights * (x - mean_unadjusted) ** 2)
    else:
        mean_unadjusted = var_rebuilt = np.nan
    checks.append({
        "event_key": event_key, "observation_date": date,
        "probability_sum": total, "bins": len(group),
        "rebuilt_variance_pp2": var_rebuilt,
        "rebuilt_mean_pp": mean_unadjusted + 0.125,
        "distribution_valid": bool(finite and np.isclose(total, 1, atol=1e-8))
    })
checks = pd.DataFrame(checks, columns=[
    "event_key", "observation_date", "probability_sum", "bins",
    "rebuilt_variance_pp2", "rebuilt_mean_pp", "distribution_valid"
])
audit = audit.merge(checks, on=["event_key", "observation_date"],
                    how="left", validate="one_to_one")
audit["variance_matches"] = np.isclose(
    audit["variance"], audit["rebuilt_variance_pp2"], atol=1e-10, rtol=1e-7
)
audit["mean_matches"] = np.isclose(
    audit["mean"], audit["rebuilt_mean_pp"], atol=1e-8, rtol=1e-7
)
audit["ust2y_signed_bps"] = audit["ust2y_change_pp"] * 100
audit["abs_ust2y_bps"] = audit["ust2y_signed_bps"].abs()
audit["kalshi_sigma_bps"] = np.sqrt(audit["variance"].clip(lower=0)) * 100

def inclusion_reason(row):
    if row["moments_match"] != "both":
        return "exclude: missing exact previous-day moments"
    if not np.isfinite(row["ust2y_signed_bps"]):
        return "exclude: missing Treasury outcome"
    if not np.isfinite(row["variance"]) or row["variance"] < -1e-12:
        return "exclude: invalid variance"
    if not (pd.notna(row["distribution_valid"]) and row["distribution_valid"]):
        return "exclude: missing or invalid distribution"
    if not row["variance_matches"] or not row["mean_matches"]:
        return "exclude: published moments do not reconcile"
    return "include: published-method candidate"

audit["inclusion_reason"] = audit.apply(inclusion_reason, axis=1)
audit["included"] = audit["inclusion_reason"].eq("include: published-method candidate")
print(audit[["meeting_date", "observation_date", "kalshi_sigma_bps",
             "abs_ust2y_bps", "inclusion_reason"]].round(3).to_string(index=False))
print("\nYear-level coverage:")
print(audit.groupby(audit["meeting_date"].dt.year).agg(
    scheduled=("included", "size"), included=("included", "sum")
).to_string())


meeting_date observation_date  kalshi_sigma_bps  abs_ust2y_bps                            inclusion_reason
  2022-01-26       2022-01-25             4.307          0.190         include: published-method candidate
  2022-03-16       2022-03-15             7.468         10.900         include: published-method candidate
  2022-05-04       2022-05-03             7.970          4.700         include: published-method candidate
  2022-06-15       2022-06-14            13.216          5.050         include: published-method candidate
  2022-07-27       2022-07-26             8.183          0.200         include: published-method candidate
  2022-09-21       2022-09-20             6.559          9.390         include: published-method candidate
  2022-11-02       2022-11-01             5.595          8.720         include: published-method candidate
  2022-12-14       2022-12-13             8.480          8.580         include: published-method candidate
  2023-02-01       2023-01-31        

### 4. Produce the candidate event dataset and report limitations
Zero constructed uncertainty is retained and flagged, not silently dropped. A conventional benchmark, revision volatility and policy-surprise definitions are later steps; this notebook does not estimate regressions.

In [4]:
dataset = audit.loc[audit["included"], [
    "meeting_date", "event_key", "observation_date", "mean",
    "kalshi_sigma_bps", "ust2y_signed_bps", "abs_ust2y_bps",
    "daily_volume", "probability_sum", "bins"
]].rename(columns={
    "mean": "authors_adjusted_mean_rate_pct",
    "daily_volume": "authors_daily_volume"
}).sort_values("meeting_date").reset_index(drop=True)

assert dataset["meeting_date"].is_unique
assert dataset["observation_date"].lt(dataset["meeting_date"]).all()
assert np.isfinite(dataset[["kalshi_sigma_bps", "abs_ust2y_bps"]]).all().all()
assert dataset["kalshi_sigma_bps"].ge(0).all()
print("Reconciled candidate events:", len(dataset))
print("Excluded scheduled events:", int((~audit["included"]).sum()))
print("Zero constructed uncertainty:", int(dataset["kalshi_sigma_bps"].eq(0).sum()))
print("\nDescriptive ranges (basis points):")
print(dataset[["kalshi_sigma_bps", "abs_ust2y_bps"]].describe().round(3).to_string())
print("\nLIMITS:")
print("- This is a dataset using the authors' processed measures, not an exact raw distribution.")
print("- Daily prices can be carried forward; daily volume does not establish age of every strike.")
print("- UTC calendar-day aggregation replaces the original 16:00 Eastern cutoff.")
print("- Reconciled means include the authors' 0.125 percentage-point adjustment.")
print("- Constructed zero variance does not establish zero true uncertainty.")
print("- Corrections, tail conventions and retrospective processing require sensitivity checks.")
print("- No claims of predictive skill or causality are made; no regressions have been run.")
print("- Policy surprise, revision volatility and conventional benchmark are not yet built.")


Reconciled candidate events: 37
Excluded scheduled events: 1
Zero constructed uncertainty: 5

Descriptive ranges (basis points):
       kalshi_sigma_bps  abs_ust2y_bps
count            37.000         37.000
mean              5.088          4.010
std               3.533          3.468
min               0.000          0.000
25%               2.512          1.040
50%               4.307          2.770
75%               7.892          6.525
max              13.216         11.775

LIMITS:
- This is a dataset using the authors' processed measures, not an exact raw distribution.
- Daily prices can be carried forward; daily volume does not establish age of every strike.
- UTC calendar-day aggregation replaces the original 16:00 Eastern cutoff.
- Reconciled means include the authors' 0.125 percentage-point adjustment.
- Constructed zero variance does not establish zero true uncertainty.
- Corrections, tail conventions and retrospective processing require sensitivity checks.
- No claims of predi

### 5. Preserve provenance and derived audit files
Only derived tables and source receipts are written to this runtime. Nothing is automatically uploaded. The three files are reproducible from the notebook and can be regenerated after a runtime reset. Saving this notebook to GitHub preserves its code and displayed aggregate/audit outputs; it does not save the runtime files.

In [5]:
# Derived outputs are written to the Colab runtime, not automatically pushed to GitHub.
OUTPUT = Path("research_outputs")
OUTPUT.mkdir(exist_ok=True)
dataset.to_csv(OUTPUT / "candidate_meeting_dataset.csv", index=False)
audit.drop(columns=["moments_match"]).to_csv(OUTPUT / "meeting_inclusion_audit.csv", index=False)
metadata = {
    "study_start": str(STUDY_START.date()),
    "as_of_date": str(AS_OF_DATE.date()),
    "sources": source_receipts,
    "method": "Diercks-Katz-Wright published daily trade-based distributions and moments",
    "timing": "Last published UTC calendar-day measure on the previous weekday; no fallback fill",
    "sigma_bps": "100 * sqrt(published variance in percentage-points squared)",
    "primary_outcome": "abs(USMPD Statements UST2Y percentage-point change * 100)",
    "mean_convention": "Published mean includes +0.125 percentage points; policy surprise deferred",
    "status": "Reconciled candidate dataset; raw staleness and processing sensitivity unresolved",
    "code_reference": "https://github.com/jdkatz21/Prediction_Markets_Public/blob/main/code/convert_trades_to_pdfs/convert_trade_level_data_cdfs.R"
}
(OUTPUT / "provenance.json").write_text(json.dumps(metadata, indent=2))
print(pd.DataFrame(source_receipts)[["source", "retrieved_utc", "sha256"]].to_string(index=False))
print("\nSaved three derived audit files under research_outputs/ in this runtime.")
print("Raw downloaded files remain in memory and are not exported.")
print("Save this notebook to ResearchProject / main as 04_research_dataset.ipynb.")
print("Next: validate the baseline and remaining predictor definitions before analysis.")


       source                    retrieved_utc                                                           sha256
      moments 2026-10-19T16:48:52.361141+00:00 539b2e016c3baabfb9a05313f2fc032d2f5a99dbade698c2df4feddd534972b9
distributions 2026-10-19T16:48:56.716140+00:00 340875fc86288f84260cd3a007bd9d0a8228e930cba3c1d86216378f0426a79a
        usmpd 2026-10-19T16:49:00.959944+00:00 f02bfcbf80cf597548d4fccb5a80b30cb1f4373a7d50e26fd1def097d43df1aa

Saved three derived audit files under research_outputs/ in this runtime.
Raw downloaded files remain in memory and are not exported.
Save this notebook to ResearchProject / main as 04_research_dataset.ipynb.
Next: validate the baseline and remaining predictor definitions before analysis.
